# Project #18 — Zero-to-Hero Notebook
## Hybrid Real-Time AI Research Assistant

This notebook explains and demonstrates a production-grade local-first hybrid RAG assistant using:
- LangGraph orchestration
- ChromaDB persistent vector storage
- Ollama local LLMs
- Optional live web search


## 1) RAG Fundamentals

Retrieval-Augmented Generation (RAG) combines retrieval and generation:
1. Retrieve relevant evidence from local/web sources
2. Build constrained context
3. Generate answers grounded in evidence

Grounding policy used in this project:
- Use context only
- If context insufficient, answer with strict fallback sentence
- Always cite sources


In [ ]:
from hybrid_research_assistant.settings import load_settings

settings = load_settings()
settings


## 2) Embeddings

Embeddings convert text into dense vectors for semantic search.

Compared models:
- BAAI/bge-small-en-v1.5 (default)
- all-MiniLM-L6-v2
- nomic-embed-text

Selection criteria:
- Retrieval quality
- Latency
- Memory footprint
- Embedding dimensionality


In [ ]:
from hybrid_research_assistant.app import build_runtime

runtime = build_runtime()
runtime.embedder.model_name


## 3) Chunking

Chunking controls retrieval granularity.

Implemented strategies:
- Recursive character chunking
- Token chunking
- Semantic chunking

Benchmark grid:
- chunk sizes: 256, 512, 768, 1024
- overlaps: 0, 50, 100, 200


In [ ]:
from hybrid_research_assistant.chunking import Chunker
from hybrid_research_assistant.schemas import ChunkingStrategy

sample_docs = runtime.loader.load_directory()[:2]
chunker = Chunker(chunk_size=512, chunk_overlap=50, strategy=ChunkingStrategy.RECURSIVE)
chunks = chunker.split_documents(sample_docs)
len(chunks)


## 4) Vector Database (ChromaDB)

ChromaDB stores embeddings and metadata for retrieval.

Implemented capabilities:
- Persistent storage
- Incremental indexing
- Document updates/deletions
- Metadata filtering
- Collection/profile scoping


In [ ]:
runtime.vector_store.count()


## 5) Similarity Search and MMR

Similarity search finds top neighbors by embedding proximity.
MMR (Maximal Marginal Relevance) improves diversity by penalizing duplicates.


In [ ]:
from hybrid_research_assistant.retrieval import RetrievalService

rows, latency_ms = runtime.retrieval.retrieve_local("What is LangGraph?", top_k=5)
len(rows), latency_ms


## 6) Hybrid Retrieval

Hybrid mode combines:
- Local Chroma retrieval
- Live web retrieval
- Merge, dedup, and rerank

This helps answer queries requiring both internal and fresh external context.


In [ ]:
# Optional live call (requires internet availability in runtime environment)
# hybrid_rows, hybrid_ms = await runtime.retrieval.retrieve_hybrid("latest AI announcement", local_k=5, web_k=5)


## 7) LangGraph Orchestration

Workflow nodes:
- intent_detection
- local/web/hybrid_retrieve
- rerank
- context_builder
- generation
- judge
- response

Error recovery node provides safe fallback behavior.


In [ ]:
diagram_path = runtime.workflow.export_diagram(settings.paths.outputs_dir / "diagrams" / "langgraph_workflow.mmd")
diagram_path


## 8) Prompt Engineering

Prompt families available:
- strict_qa
- research_assistant
- teacher
- technical_mentor
- summarizer

All prompts enforce strict grounding and citation requirements.


In [ ]:
response = runtime.workflow.ask("What is retrieval-augmented generation?", prompt_name="teacher")
response.answer[:500]


## 9) Evaluation and LLM-as-a-Judge

Evaluation dimensions:
- Accuracy
- Faithfulness
- Relevance
- Context precision
- Latency

Judge model:
- granite4.1:3b


In [ ]:
from hybrid_research_assistant.evaluation import load_eval_samples, evaluate_benchmark

samples = load_eval_samples(settings.evaluation.benchmark_path)
len(samples)


## 10) End-to-End Demo

Run one complete query through LangGraph pipeline and inspect:
- route mode
- answer
- citations
- latency breakdown
- judge scores


In [ ]:
from hybrid_research_assistant.schemas import RetrievalMode

demo = runtime.workflow.ask(
    "Compare recent AI announcements with our internal policy constraints.",
    mode=RetrievalMode.LOCAL,
    prompt_name="research_assistant",
)
{
    "mode": demo.mode.value,
    "reason": demo.route_reason,
    "answer_preview": demo.answer[:350],
    "citations": [c.model_dump() if hasattr(c, "model_dump") else {"source_file": c.source_file, "page_number": c.page_number, "url": c.url, "chunk_id": c.chunk_id, "confidence": c.confidence, "title": c.title} for c in demo.citations[:3]],
    "timings_ms": {"intent_ms": demo.timings.intent_ms, "retrieval_ms": demo.timings.retrieval_ms, "rerank_ms": demo.timings.rerank_ms, "context_ms": demo.timings.context_ms, "generation_ms": demo.timings.generation_ms, "judge_ms": demo.timings.judge_ms, "total_ms": demo.timings.total_ms},
    "judge": demo.judge,
}
